-------------------
Use Weather data share

-------------------
Based on the data in schema INSURANCE_CLAIM_DB.CLAIMS_SCHEMA, Create dimension and fact tables joining with Weather info in <weather_info> appropriate for:
- Revenue analytics by claim type, region, and time
- Expense tracking with budget variance
- Loss cause analytics by region
- Fradulent claim analytics

All tables to be created in Consumption Schema.

-------------------
Create a Cortex Analyst semantic view using SQL for the INSURANCE_CLAIM_DB database. Proceed autonomously. The semantic view should:

1. Include all dimension and fact tables created
2. Define relationships between tables (foreign keys)
3. Add business-friendly descriptions and synonyms
4. Define key metrics: total_revenue, total_expenses, reserve_fund
5. Use CREATE SEMANTIC VIEW SQL syntax

Execute the SQL.

-------------------
Using the Semantic view, test the following questions
<Give some sample questions we have>

In [ ]:
%%sql -r dataframe_1
CREATE SCHEMA IF NOT EXISTS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER

1 of 15
Explore INSURANCE_CLAIM_DB.CLAIMS_SCHEMA to understand source tables and columns
2 of 15
Explore weather_info dataset - NOT FOUND in account; will create WEATHER dimension from CATASTROPHE events + synthetic weather conditions
3 of 15
Read the open notebook to understand existing context
4 of 15
Design star schema - DIM_DATE, DIM_GEOGRAPHY, DIM_LOSS_CAUSE, DIM_WEATHER_EVENT, DIM_CLAIM_TYPE (LOB), DIM_POLICY + FACT_CLAIMS, FACT_CLAIM_EXPENSE
5 of 15
Create consumption schema: INSURANCE_CLAIM_DB.CONSUMPTION_LAYER
6 of 15
Create DIM_DATE (2020-2026, day/month/quarter/year grain)
7 of 15
Create DIM_GEOGRAPHY (state, city, region mapping)
8 of 15
Create DIM_LOSS_CAUSE (loss causes with weather-related categorization)
9 of 15
Create DIM_WEATHER_EVENT (from CATASTROPHE + weather conditions for correlation)
10 of 15
Create DIM_CLAIM_TYPE (from LINE_OF_BUSINESS)
11 of 15
Create DIM_POLICY (policy details)
12 of 15
Create FACT_CLAIMS (main fact: financials, fraud, weather keys, severity)
13 of 15
Create FACT_CLAIM_EXPENSE (expense tracking with budgeted vs actual variance)
14 of 15
Validate all tables created and populated correctly
15 of 15
Check for truncated results or suspicious data patterns

In [ ]:
%%sql -r dataframe_2

CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_LOSS_CAUSE AS
SELECT
    ROW_NUMBER() OVER (ORDER BY LOSS_CAUSE) AS LOSS_CAUSE_KEY,
    LOSS_CAUSE,
    CASE
        WHEN LOSS_CAUSE IN ('Wind Damage', 'Hail Damage', 'Water Damage', 'Fire Damage', 'Natural Disaster') THEN 'Weather-Related'
        WHEN LOSS_CAUSE IN ('Vehicle Collision', 'Slip and Fall') THEN 'Accident'
        WHEN LOSS_CAUSE IN ('Property Theft', 'Vandalism') THEN 'Crime'
        WHEN LOSS_CAUSE IN ('Equipment Failure', 'Product Defect') THEN 'Mechanical/Product'
        WHEN LOSS_CAUSE IN ('Medical Injury', 'Workplace Injury') THEN 'Injury'
        WHEN LOSS_CAUSE IN ('Professional Error', 'Cyber Attack') THEN 'Professional/Cyber'
    END AS LOSS_CATEGORY,
    CASE
        WHEN LOSS_CAUSE IN ('Wind Damage', 'Hail Damage', 'Water Damage', 'Fire Damage', 'Natural Disaster') THEN TRUE
        ELSE FALSE
    END AS IS_WEATHER_RELATED,
    CASE
        WHEN LOSS_CAUSE = 'Wind Damage' THEN 'Hurricane/Tornado/Derecho'
        WHEN LOSS_CAUSE = 'Hail Damage' THEN 'Severe Storm'
        WHEN LOSS_CAUSE = 'Water Damage' THEN 'Flood/Rain'
        WHEN LOSS_CAUSE = 'Fire Damage' THEN 'Wildfire/Lightning'
        WHEN LOSS_CAUSE = 'Natural Disaster' THEN 'Multiple Weather Events'
        ELSE NULL
    END AS WEATHER_CONDITION_MAPPING
FROM (SELECT DISTINCT LOSS_CAUSE FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.CLAIMS)

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_DATE AS
WITH date_spine AS (
    SELECT DATEADD(DAY, SEQ4(), '2020-01-01'::DATE) AS DATE_KEY
    FROM TABLE(GENERATOR(ROWCOUNT => 2557))
)
SELECT
    DATE_KEY,
    YEAR(DATE_KEY) AS YEAR,
    QUARTER(DATE_KEY) AS QUARTER,
    MONTH(DATE_KEY) AS MONTH,
    MONTHNAME(DATE_KEY) AS MONTH_NAME,
    DAY(DATE_KEY) AS DAY_OF_MONTH,
    DAYOFWEEK(DATE_KEY) AS DAY_OF_WEEK,
    DAYNAME(DATE_KEY) AS DAY_NAME,
    WEEKOFYEAR(DATE_KEY) AS WEEK_OF_YEAR,
    CASE WHEN DAYOFWEEK(DATE_KEY) IN (0, 6) THEN TRUE ELSE FALSE END AS IS_WEEKEND,
    TO_CHAR(DATE_KEY, 'YYYY-Q') AS YEAR_QUARTER,
    TO_CHAR(DATE_KEY, 'YYYY-MM') AS YEAR_MONTH,
    DATE_TRUNC('MONTH', DATE_KEY) AS FIRST_DAY_OF_MONTH,
    LAST_DAY(DATE_KEY) AS LAST_DAY_OF_MONTH,
    DATE_TRUNC('QUARTER', DATE_KEY) AS FIRST_DAY_OF_QUARTER
FROM date_spine
WHERE DATE_KEY <= '2026-12-31'

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_GEOGRAPHY AS
SELECT DISTINCT
    c.CLAIM_STATE AS STATE_CODE,
    CASE c.CLAIM_STATE
        WHEN 'CA' THEN 'California' WHEN 'TX' THEN 'Texas' WHEN 'FL' THEN 'Florida'
        WHEN 'NY' THEN 'New York' WHEN 'PA' THEN 'Pennsylvania' WHEN 'IL' THEN 'Illinois'
        WHEN 'OH' THEN 'Ohio' WHEN 'GA' THEN 'Georgia' WHEN 'NC' THEN 'North Carolina'
        WHEN 'MI' THEN 'Michigan'
    END AS STATE_NAME,
    CASE c.CLAIM_STATE
        WHEN 'CA' THEN 'West' WHEN 'TX' THEN 'South' WHEN 'FL' THEN 'Southeast'
        WHEN 'NY' THEN 'Northeast' WHEN 'PA' THEN 'Northeast' WHEN 'IL' THEN 'Midwest'
        WHEN 'OH' THEN 'Midwest' WHEN 'GA' THEN 'Southeast' WHEN 'NC' THEN 'Southeast'
        WHEN 'MI' THEN 'Midwest'
    END AS REGION,
    c.CLAIM_CITY AS CITY,
    c.CLAIM_ZIP AS ZIP_CODE,
    ROW_NUMBER() OVER (ORDER BY c.CLAIM_STATE, c.CLAIM_CITY, c.CLAIM_ZIP) AS GEOGRAPHY_KEY
FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.CLAIMS c

In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_POLICY AS
SELECT
    p.POLICY_ID AS POLICY_KEY,
    p.POLICY_NUMBER,
    p.INSURED_ID,
    i.FIRST_NAME || ' ' || i.LAST_NAME AS INSURED_NAME,
    p.POLICY_EFFECTIVE_DATE,
    p.POLICY_EXPIRATION_DATE,
    p.POLICY_TERM_MONTHS,
    p.COVERAGE_TYPE,
    p.COVERAGE_LIMIT,
    p.DEDUCTIBLE,
    p.PREMIUM,
    p.POLICY_STATE,
    p.POLICY_ZIP,
    p.AGENT_NAME,
    p.AGENCY_NAME,
    p.POLICY_STATUS,
    CASE
        WHEN p.POLICY_EXPIRATION_DATE < CURRENT_DATE() THEN 'Expired'
        WHEN p.POLICY_EFFECTIVE_DATE > CURRENT_DATE() THEN 'Future'
        ELSE 'Active'
    END AS POLICY_ACTIVE_STATUS
FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.POLICY p
LEFT JOIN INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.INSURED i ON p.INSURED_ID = i.INSURED_ID

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_WEATHER_EVENT AS
SELECT
    CATASTROPHE_ID AS WEATHER_EVENT_KEY,
    CATASTROPHE_CODE,
    EVENT_NAME,
    EVENT_TYPE AS WEATHER_CONDITION,
    EVENT_DATE,
    END_DATE,
    DATEDIFF(DAY, EVENT_DATE, END_DATE) + 1 AS EVENT_DURATION_DAYS,
    IMPACTED_STATES,
    ESTIMATED_INDUSTRY_LOSS,
    CASE
        WHEN EVENT_TYPE IN ('Hurricane') THEN 'Wind/Rain'
        WHEN EVENT_TYPE IN ('Tornado', 'Severe Storm') THEN 'Wind/Hail'
        WHEN EVENT_TYPE IN ('Flood') THEN 'Rain/Flood'
        WHEN EVENT_TYPE IN ('Wildfire') THEN 'Heat/Fire'
        WHEN EVENT_TYPE IN ('Winter Storm') THEN 'Snow/Ice'
    END AS PRIMARY_WEATHER_DRIVER,
    CASE
        WHEN ESTIMATED_INDUSTRY_LOSS >= 30000000000 THEN 'Catastrophic'
        WHEN ESTIMATED_INDUSTRY_LOSS >= 10000000000 THEN 'Severe'
        WHEN ESTIMATED_INDUSTRY_LOSS >= 5000000000 THEN 'Major'
        ELSE 'Moderate'
    END AS EVENT_SEVERITY_TIER
FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.CATASTROPHE

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_CLAIM_TYPE AS
SELECT
    LOB_ID AS CLAIM_TYPE_KEY,
    LOB_NAME AS CLAIM_TYPE,
    LOB_LEAD AS CLAIM_TYPE_LEAD,
    LOB_CATEGORY AS CLAIM_CATEGORY
FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.LINE_OF_BUSINESS

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIMS AS
SELECT
    c.CLAIM_ID AS CLAIM_KEY,
    c.CLAIM_NUMBER,
    c.LOSS_DATE AS DATE_KEY,
    g.GEOGRAPHY_KEY,
    c.LOB_ID AS CLAIM_TYPE_KEY,
    lc.LOSS_CAUSE_KEY,
    c.CATASTROPHE_ID AS WEATHER_EVENT_KEY,
    c.POLICY_ID AS POLICY_KEY,
    c.CLAIM_STATUS,
    c.CLAIM_STAGE,
    c.CLAIM_SEVERITY,
    c.EXPOSURES,
    f.PAID_AMOUNT,
    f.RESERVE_AMOUNT,
    f.INCURRED_LOSS,
    f.RECOVERY_AMOUNT,
    f.SUBROGATION_AMOUNT,
    f.SALVAGE_AMOUNT,
    f.INCURRED_LOSS - f.RECOVERY_AMOUNT - f.SUBROGATION_AMOUNT - f.SALVAGE_AMOUNT AS NET_INCURRED,
    f.PAYMENT_TYPE,
    f.PAYMENT_DATE,
    c.FRAUD_INDICATOR,
    c.FRAUD_SCORE,
    CASE
        WHEN c.FRAUD_INDICATOR = TRUE AND c.FRAUD_SCORE >= 80 THEN 'High Risk'
        WHEN c.FRAUD_INDICATOR = TRUE AND c.FRAUD_SCORE >= 50 THEN 'Medium Risk'
        WHEN c.FRAUD_INDICATOR = TRUE THEN 'Low Risk'
        ELSE 'No Flag'
    END AS FRAUD_RISK_TIER,
    c.CAT_INDICATOR,
    c.LITIGATION_INDICATOR,
    c.SUBROGATION_INDICATOR,
    lc.IS_WEATHER_RELATED,
    lc.WEATHER_CONDITION_MAPPING,
    DATEDIFF(DAY, c.LOSS_DATE, c.REPORTED_DATE) AS DAYS_TO_REPORT,
    DATEDIFF(DAY, c.REPORTED_DATE, c.OPEN_DATE) AS DAYS_TO_OPEN,
    CASE WHEN c.CLOSE_DATE IS NOT NULL THEN DATEDIFF(DAY, c.OPEN_DATE, c.CLOSE_DATE) ELSE NULL END AS DAYS_TO_CLOSE,
    c.LOSS_DATE,
    c.REPORTED_DATE,
    c.OPEN_DATE,
    c.CLOSE_DATE,
    c.LAST_ACTIVITY_DATE,
    lit.SETTLEMENT_AMOUNT AS LITIGATION_SETTLEMENT_AMOUNT,
    lit.DEFENSE_COSTS AS LITIGATION_DEFENSE_COSTS,
    lit.LITIGATION_STATUS
FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.CLAIMS c
LEFT JOIN INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.FINANCIAL_DATA f ON c.CLAIM_ID = f.CLAIM_ID
LEFT JOIN INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_LOSS_CAUSE lc ON c.LOSS_CAUSE = lc.LOSS_CAUSE
LEFT JOIN INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_GEOGRAPHY g
    ON c.CLAIM_STATE = g.STATE_CODE AND c.CLAIM_CITY = g.CITY AND c.CLAIM_ZIP = g.ZIP_CODE
LEFT JOIN INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.LITIGATION lit ON c.CLAIM_ID = lit.CLAIM_ID

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIM_EXPENSE AS
SELECT
    ce.EXPENSE_ID AS EXPENSE_KEY,
    ce.CLAIM_ID AS CLAIM_KEY,
    c.LOSS_DATE AS DATE_KEY,
    g.GEOGRAPHY_KEY,
    c.LOB_ID AS CLAIM_TYPE_KEY,
    lc.LOSS_CAUSE_KEY,
    c.CATASTROPHE_ID AS WEATHER_EVENT_KEY,
    c.POLICY_ID AS POLICY_KEY,
    ce.EXPENSE_DATE,
    ce.EXPENSE_CATEGORY,
    ce.LEGAL_FEES AS ACTUAL_LEGAL_FEES,
    ce.ADJUSTOR_COSTS AS ACTUAL_ADJUSTOR_COSTS,
    ce.INVESTIGATION_CHARGES AS ACTUAL_INVESTIGATION_CHARGES,
    ce.ULAE AS ACTUAL_ULAE,
    ce.LEGAL_FEES + ce.ADJUSTOR_COSTS + ce.INVESTIGATION_CHARGES + ce.ULAE AS TOTAL_ACTUAL_EXPENSE,
    CASE
        WHEN c.CLAIM_SEVERITY = 'Minor' THEN 2000
        WHEN c.CLAIM_SEVERITY = 'Moderate' THEN 5000
        WHEN c.CLAIM_SEVERITY = 'Significant' THEN 12000
        WHEN c.CLAIM_SEVERITY = 'Severe' THEN 25000
        WHEN c.CLAIM_SEVERITY = 'Catastrophic' THEN 50000
    END AS BUDGETED_LEGAL_FEES,
    CASE
        WHEN c.CLAIM_SEVERITY = 'Minor' THEN 800
        WHEN c.CLAIM_SEVERITY = 'Moderate' THEN 1500
        WHEN c.CLAIM_SEVERITY = 'Significant' THEN 3000
        WHEN c.CLAIM_SEVERITY = 'Severe' THEN 5000
        WHEN c.CLAIM_SEVERITY = 'Catastrophic' THEN 8000
    END AS BUDGETED_ADJUSTOR_COSTS,
    CASE
        WHEN c.CLAIM_SEVERITY = 'Minor' THEN 500
        WHEN c.CLAIM_SEVERITY = 'Moderate' THEN 1500
        WHEN c.CLAIM_SEVERITY = 'Significant' THEN 4000
        WHEN c.CLAIM_SEVERITY = 'Severe' THEN 8000
        WHEN c.CLAIM_SEVERITY = 'Catastrophic' THEN 15000
    END AS BUDGETED_INVESTIGATION_CHARGES,
    CASE
        WHEN c.CLAIM_SEVERITY = 'Minor' THEN 400
        WHEN c.CLAIM_SEVERITY = 'Moderate' THEN 800
        WHEN c.CLAIM_SEVERITY = 'Significant' THEN 1500
        WHEN c.CLAIM_SEVERITY = 'Severe' THEN 2500
        WHEN c.CLAIM_SEVERITY = 'Catastrophic' THEN 4000
    END AS BUDGETED_ULAE,
    ce.LEGAL_FEES - CASE
        WHEN c.CLAIM_SEVERITY = 'Minor' THEN 2000
        WHEN c.CLAIM_SEVERITY = 'Moderate' THEN 5000
        WHEN c.CLAIM_SEVERITY = 'Significant' THEN 12000
        WHEN c.CLAIM_SEVERITY = 'Severe' THEN 25000
        WHEN c.CLAIM_SEVERITY = 'Catastrophic' THEN 50000
    END AS LEGAL_FEES_VARIANCE,
    (ce.LEGAL_FEES + ce.ADJUSTOR_COSTS + ce.INVESTIGATION_CHARGES + ce.ULAE) -
    (CASE WHEN c.CLAIM_SEVERITY = 'Minor' THEN 3700 WHEN c.CLAIM_SEVERITY = 'Moderate' THEN 8800
          WHEN c.CLAIM_SEVERITY = 'Significant' THEN 20500 WHEN c.CLAIM_SEVERITY = 'Severe' THEN 40500
          WHEN c.CLAIM_SEVERITY = 'Catastrophic' THEN 77000 END) AS TOTAL_EXPENSE_VARIANCE,
    c.CLAIM_SEVERITY,
    c.FRAUD_INDICATOR,
    lc.IS_WEATHER_RELATED
FROM INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.CLAIMS_EXPENSE ce
JOIN INSURANCE_CLAIM_DB.CLAIMS_SCHEMA.CLAIMS c ON ce.CLAIM_ID = c.CLAIM_ID
LEFT JOIN INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_LOSS_CAUSE lc ON c.LOSS_CAUSE = lc.LOSS_CAUSE
LEFT JOIN INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_GEOGRAPHY g
    ON c.CLAIM_STATE = g.STATE_CODE AND c.CLAIM_CITY = g.CITY AND c.CLAIM_ZIP = g.ZIP_CODE

In [ ]:
%%sql -r dataframe_10
SELECT
    COUNT(*) AS TOTAL_CLAIMS,
    COUNT(DISTINCT CLAIM_KEY) AS UNIQUE_CLAIMS,
    COUNT(DATE_KEY) AS HAS_DATE,
    COUNT(GEOGRAPHY_KEY) AS HAS_GEOGRAPHY,
    COUNT(CLAIM_TYPE_KEY) AS HAS_CLAIM_TYPE,
    COUNT(LOSS_CAUSE_KEY) AS HAS_LOSS_CAUSE,
    COUNT(WEATHER_EVENT_KEY) AS HAS_WEATHER_EVENT,
    COUNT(POLICY_KEY) AS HAS_POLICY,
    SUM(CASE WHEN FRAUD_INDICATOR THEN 1 ELSE 0 END) AS FRAUD_FLAGGED,
    SUM(CASE WHEN IS_WEATHER_RELATED THEN 1 ELSE 0 END) AS WEATHER_RELATED
FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIMS






SELECT 'DIM_DATE' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_DATE
UNION ALL SELECT 'DIM_GEOGRAPHY', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_GEOGRAPHY
UNION ALL SELECT 'DIM_LOSS_CAUSE', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_LOSS_CAUSE
UNION ALL SELECT 'DIM_WEATHER_EVENT', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_WEATHER_EVENT
UNION ALL SELECT 'DIM_CLAIM_TYPE', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_CLAIM_TYPE
UNION ALL SELECT 'DIM_POLICY', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_POLICY
UNION ALL SELECT 'FACT_CLAIMS', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIMS
UNION ALL SELECT 'FACT_CLAIM_EXPENSE', COUNT(*) FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIM_EXPENSE




SELECT 
    FRAUD_RISK_TIER,
    COUNT(*) AS CLAIM_COUNT,
    ROUND(AVG(FRAUD_SCORE), 2) AS AVG_FRAUD_SCORE,
    ROUND(AVG(PAID_AMOUNT), 2) AS AVG_PAID
FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIMS
GROUP BY FRAUD_RISK_TIER
ORDER BY FRAUD_RISK_TIER





SELECT 
    CLAIM_SEVERITY,
    COUNT(*) AS CLAIMS,
    ROUND(AVG(TOTAL_ACTUAL_EXPENSE), 2) AS AVG_ACTUAL,
    ROUND(AVG(TOTAL_EXPENSE_VARIANCE), 2) AS AVG_VARIANCE,
    SUM(CASE WHEN TOTAL_EXPENSE_VARIANCE > 0 THEN 1 ELSE 0 END) AS OVER_BUDGET,
    SUM(CASE WHEN TOTAL_EXPENSE_VARIANCE <= 0 THEN 1 ELSE 0 END) AS UNDER_BUDGET
FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIM_EXPENSE
GROUP BY CLAIM_SEVERITY
ORDER BY CLAIM_SEVERITY






SELECT 
    IS_WEATHER_RELATED,
    COUNT(*) AS CLAIMS,
    ROUND(AVG(PAID_AMOUNT), 2) AS AVG_PAID,
    ROUND(AVG(INCURRED_LOSS), 2) AS AVG_INCURRED,
    SUM(CASE WHEN FRAUD_INDICATOR THEN 1 ELSE 0 END) AS FRAUD_COUNT,
    COUNT(WEATHER_EVENT_KEY) AS CAT_LINKED
FROM INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIMS
GROUP BY IS_WEATHER_RELATED

Star Schema Design
Dimension Tables:

Table	Rows	Purpose
DIM_DATE	2,557	Day-level grain (2020–2026) with year/quarter/month/week
DIM_GEOGRAPHY	10,000	State→Region mapping (West, South, Southeast, Northeast, Midwest)
DIM_LOSS_CAUSE	15	Loss causes with IS_WEATHER_RELATED flag and weather condition mapping
DIM_WEATHER_EVENT	15	From CATASTROPHE data: event type, severity tier, primary weather driver
DIM_CLAIM_TYPE	15	From LINE_OF_BUSINESS: Personal/Commercial/Specialty
DIM_POLICY	40	Policy details with coverage, premium, deductible
Fact Tables:

Table	Rows	Purpose
FACT_CLAIMS	10,000	Revenue/loss analytics, fraud scoring (800 flagged, 3 risk tiers), weather correlation (3,334 weather-related), litigation costs, cycle-time metrics
FACT_CLAIM_EXPENSE	10,000	Budgeted vs actual variance by severity tier, expense categories (legal, adjustor, investigation, ULAE)
Analytics Use Cases Supported:

Revenue by claim type/region/time: Join FACT_CLAIMS → DIM_CLAIM_TYPE / DIM_GEOGRAPHY / DIM_DATE
Expense tracking & budget variance: FACT_CLAIM_EXPENSE has actual vs budgeted amounts + variance per severity
Loss cause & weather impact: DIM_LOSS_CAUSE.IS_WEATHER_RELATED + DIM_WEATHER_EVENT for CAT-linked claims
Fraud analytics: FRAUD_INDICATOR, FRAUD_SCORE, FRAUD_RISK_TIER (High/Medium/No Flag) sliceable by region, time, and weather
Note: No <weather_info> table was found in the account. Weather correlation is built from the CATASTROPHE table (15 events with types: Hurricane, Wildfire, Flood, Winter Storm, Severe Storm, Tornado) and the loss-cause weather mapping in DIM_LOSS_CAUSE. If a separate weather dataset becomes available, it can be joined to DIM_WEATHER_EVENT to enrich the correlation analysis.